# BRGI Datathon 2026 — Quantile Forecasting Baseline

This notebook trains three CatBoost quantile-regression models for **p10, p50, p90** and writes `submission.csv` in the exact sample-submission format. It is designed for Kaggle or a local Python environment with `catboost`, `pandas`, and `numpy`.

**Validation:** hold out later `origin_quarter` values to approximate forecasting into the future. Review the validation results before trusting the final predictions. The test set contains districts not represented in training, so we avoid relying on district-specific target averages.

> Important: confirm the competition rules and metric before final tuning. This is a strong, reproducible baseline, not a guarantee of a leaderboard score.

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor

SEED = 42
TRAIN_PATH = "/kaggle/input/brgi-datathon-2026/train.csv"
TEST_PATH = "/kaggle/input/brgi-datathon-2026/test.csv"
SAMPLE_PATH = "/kaggle/input/brgi-datathon-2026/sample_submission.csv"

# For local use, replace the paths above with the locations of your CSV files.
if not os.path.exists(TRAIN_PATH): TRAIN_PATH = "train.csv"
if not os.path.exists(TEST_PATH): TEST_PATH = "test.csv"
if not os.path.exists(SAMPLE_PATH): SAMPLE_PATH = "sample_submission.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample = pd.read_csv(SAMPLE_PATH)
print("train:", train.shape, "test:", test.shape, "sample:", sample.shape)
print("target range:", train.brgi.min(), train.brgi.max())
print("test quarters:", sorted(test.target_quarter.unique()))
print("submission columns:", sample.columns.tolist())

## 1. Feature preparation

We exclude the target and row identifier. `district_id` is retained as a categorical input, but the model must still generalize to unseen category values. Missing numerical values are handled by CatBoost. Quarter strings are categorical features.

In [ ]:
TARGET = "brgi"
ID_COL = "id"
CAT_COLS = [c for c in ["district_id", "state", "region", "origin_quarter", "target_quarter"] if c in train.columns]
DROP_COLS = [TARGET, ID_COL]
FEATURES = [c for c in train.columns if c not in DROP_COLS]

def prepare_features(df):
    X = df[FEATURES].copy()
    for c in CAT_COLS:
        X[c] = X[c].fillna("__MISSING__").astype(str)
    # CatBoost accepts NaNs in numeric columns. Convert any accidental object columns to strings.
    for c in X.columns:
        if c not in CAT_COLS and X[c].dtype == "object":
            X[c] = X[c].fillna("__MISSING__").astype(str)
            CAT_COLS.append(c)
    return X

X_all = prepare_features(train)
X_test = prepare_features(test)
y_all = train[TARGET].astype(float)
cat_indices = [X_all.columns.get_loc(c) for c in CAT_COLS if c in X_all.columns]
print("features:", len(FEATURES), "categorical:", CAT_COLS)

## 2. Time-aware validation

Use later origin quarters as a holdout. This is more realistic than a random split because neighboring district-quarter rows are highly related. The threshold is configurable; use the latest available origin quarters for validation and ensure every horizon is represented where possible.

In [ ]:
# Parse quarter strings like 2023Q4 into sortable integers.
def quarter_code(s):
    s = str(s)
    return int(s[:4]) * 4 + (int(s[-1]) - 1)

origin_codes = train["origin_quarter"].map(quarter_code)
unique_origins = sorted(train["origin_quarter"].dropna().unique(), key=quarter_code)
# Keep the latest ~20% of distinct origin quarters for validation, with a minimum of 4.
val_n = min(max(4, int(np.ceil(len(unique_origins) * 0.20))), max(1, len(unique_origins)-1))
val_origins = unique_origins[-val_n:]
val_mask = train["origin_quarter"].isin(val_origins)
tr_idx = np.flatnonzero(~val_mask.to_numpy())
va_idx = np.flatnonzero(val_mask.to_numpy())
print("validation origin quarters:", val_origins)
print("train/validation rows:", len(tr_idx), len(va_idx))
print("validation horizons:\n", train.loc[val_mask, "horizon"].value_counts().sort_index())

## 3. Train p10, p50, p90 models

CatBoost's `Quantile:alpha=...` objective directly estimates the requested quantile. We use early stopping on the temporal holdout and report pinball loss. The validation split is used for diagnostics; final models are then refit on all training rows.

In [ ]:
def pinball_loss(y_true, y_pred, alpha):
    err = np.asarray(y_true) - np.asarray(y_pred)
    return float(np.mean(np.maximum(alpha * err, (alpha - 1) * err)))

quantiles = {"p10": 0.10, "p50": 0.50, "p90": 0.90}
validation_predictions = {}
validation_models = {}
for name, alpha in quantiles.items():
    model = CatBoostRegressor(
        loss_function=f"Quantile:alpha={alpha}",
        eval_metric=f"Quantile:alpha={alpha}",
        iterations=2500, learning_rate=0.035, depth=8,
        l2_leaf_reg=5, random_seed=SEED,
        verbose=250, early_stopping_rounds=150,
        allow_writing_files=False,
    )
    model.fit(
        X_all.iloc[tr_idx], y_all.iloc[tr_idx],
        cat_features=cat_indices,
        eval_set=(X_all.iloc[va_idx], y_all.iloc[va_idx]),
        use_best_model=True,
    )
    pred = model.predict(X_all.iloc[va_idx])
    validation_predictions[name] = pred
    validation_models[name] = model
    print(name, "best_iteration=", model.get_best_iteration(),
          "pinball_loss=", round(pinball_loss(y_all.iloc[va_idx], pred, alpha), 5))

In [ ]:
# Useful diagnostics: central estimate MAE and interval coverage.
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
y_val = y_all.iloc[va_idx].to_numpy()
p10v = validation_predictions["p10"]
p50v = validation_predictions["p50"]
p90v = validation_predictions["p90"]
print("p50 MAE:", round(mean_absolute_error(y_val, p50v), 4))
print("p50 RMSE:", round(mean_squared_error(y_val, p50v) ** 0.5, 4))
print("p50 R2:", round(r2_score(y_val, p50v), 4))
print("80% interval coverage:", round(float(np.mean((y_val >= p10v) & (y_val <= p90v))), 4))
print("quantile crossing before correction:", int(np.sum((p10v > p50v) | (p50v > p90v))))

## 4. Fit final models and create submission

Quantile predictions can occasionally cross. We sort the three predictions row-wise to enforce `p10 <= p50 <= p90`. This guarantees valid ordering but does not itself guarantee perfect statistical calibration.

In [ ]:
final_models = {}
test_predictions = {}
for name, alpha in quantiles.items():
    best_iter = validation_models[name].get_best_iteration()
    iterations = best_iter + 1 if best_iter is not None and best_iter > 0 else 1200
    model = CatBoostRegressor(
        loss_function=f"Quantile:alpha={alpha}",
        iterations=iterations, learning_rate=0.035, depth=8,
        l2_leaf_reg=5, random_seed=SEED,
        verbose=250, allow_writing_files=False,
    )
    model.fit(X_all, y_all, cat_features=cat_indices)
    final_models[name] = model
    test_predictions[name] = model.predict(X_test)

pred_matrix = np.column_stack([test_predictions["p10"], test_predictions["p50"], test_predictions["p90"]])
pred_matrix = np.sort(pred_matrix, axis=1)
# BRGI observed in training ranges from 0.5 to 98.25; clip to observed support as a conservative guardrail.
y_min, y_max = float(y_all.min()), float(y_all.max())
pred_matrix = np.clip(pred_matrix, y_min, y_max)

submission = sample.copy()
submission_ids = submission["id"].astype(str).to_numpy()
test_ids = test["id"].astype(str).to_numpy()
assert np.array_equal(submission_ids, test_ids), "sample_submission and test IDs/order do not match"
for j, col in enumerate(["p10", "p50", "p90"]):
    submission[col] = pred_matrix[:, j]
assert submission[["p10", "p50", "p90"]].notna().all().all()
assert (submission.p10 <= submission.p50).all() and (submission.p50 <= submission.p90).all()
submission.to_csv("submission.csv", index=False)
print("Saved submission.csv", submission.shape)
display(submission.head())

## 5. Optional feature importance

Use the median model for a first interpretation. Feature importance is descriptive, not causal.

In [ ]:
importance = pd.DataFrame({
    "feature": FEATURES,
    "importance": final_models["p50"].get_feature_importance(),
}).sort_values("importance", ascending=False)
display(importance.head(25))